# 一、序列模型
## 1.1 理论计算题
给定字符序列 `ababc`，采用一阶马尔可夫模型 $p(x_t | x_{t-1})$，使用拉普拉斯平滑（加1平滑）估计以下条件概率：
1. $p(a|b)$
2. $p(c|b)$     
(词汇表为 {' a', b', c'}，计算时考虑所有可能转移，包括未出现的情况｡)



一阶马尔可夫假设：  
$p(x_1, x_2, \dots, x_T) = p(x_1) \prod_{t=2}^T p(x_t \mid x_{t-1})$

统计序列 `"ababc"` 中的转移次数：

| 转移 | 次数 |
|------|------|
| a→b  | 2    |
| b→a  | 1    |
| b→c  | 1    |
| 其他 | 0    |

词汇表大小 \(V = 3\)，\(\text{count}(b) = 2\)（b出现于位置2和4）。

拉普拉斯平滑公式：  
$$
p(x' \mid b) = \frac{\text{count}(b, x') + 1}{\text{count}(b) + V}
$$

1. $p(\text{a} \mid \text{b}) = \frac{1+1}{2+3} = \frac{2}{5} = 0.4$
2. $p(\text{c} \mid \text{b}) = \frac{1+1}{2+3} = \frac{2}{5} = 0.4$


### 1.2 编程题
编写一个函数preprocess_text(text, n)，完成以下步骤：
1. 将文本转换为小写，去除标点符号（保留字母和空格）。
2. 按空格分词。
3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）。
4. 用滑动窗口生成长度为 $n$ 的特征序列和对应的下一个词标签（用于自回归语言模型）。

返回词汇表字典和(特征列表, 标签列表)。例如，输入"The time machine" 和 $n=2$，应生成特征[['the','time'], ['time','machine']] 和标签['machine', None]（若无后续词则忽略）。


In [15]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去标点（仅保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    # 2. 按空格分词
    tokens = text.split()
    # 3. 构建词汇表：按词频降序，同频按首次出现顺序
    freq = Counter(tokens)
    first_occur = {w: i for i, w in enumerate(tokens)}
    sorted_vocab = sorted(freq.keys(), key=lambda w: (-freq[w], first_occur[w]))
    word_to_id = {w: i for i, w in enumerate(sorted_vocab)}
    # 4. 滑动窗口生成 n-gram 特征及下一个词标签
    features, labels = [], []
    for i in range(len(tokens) - n + 1):
        features.append(tokens[i:i+n])
        labels.append(tokens[i+n] if i+n < len(tokens) else None)
    return word_to_id, (features, labels)



In [16]:
# 示例
text = "The time machine"
vocab, (feat, lab) = preprocess_text(text, n=2)
print("词汇表:", vocab)
print("特征:", feat)
print("标签:", lab)

词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征: [['the', 'time'], ['time', 'machine']]
标签: ['machine', None]



## 二、 循环神经网络
### 2.1 理论计算题
考虑一个线性RNN（无偏置），定义为 $h_{t}=W_{h h} h_{t-1}+W_{h x} x_{t}$，输出 $o_{t}=W_{o h} h_{t}$。假设损失函数为平方损失 $L=\frac{1}{2} \sum_{t=1}^{T}(o_{t}-y_{t})^{2}$。推导损失对权重 $W_{h h}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。


**已知条件**：
1. 前向传播
$$
\begin{cases}
h_t = W_{hh} h_{t-1} + W_{hx} x_t \\
o_t = W_{oh} h_t
\end{cases}
$$
2. 整体平方损失
$$
L = \frac12 \sum_{t=1}^T \|o_t - y_t\|_2^2
$$
3. 目标：求 $\displaystyle \frac{\partial L}{\partial W_{hh}}$

**步骤1：单时间步损失梯度 $\dfrac{\partial L}{\partial o_t}$**
记单步损失 $L_t = \dfrac12\|o_t-y_t\|^2$，$L=\sum L_t$
$$
\frac{\partial L_t}{\partial o_t} = o_t - y_t,\quad \frac{\partial L}{\partial o_t} = o_t - y_t
$$

**步骤2：损失对隐藏状态 $h_t$ 的梯度（递归BPTT核心）**
由链式法则：
$$
\frac{\partial L}{\partial h_t}
= \frac{\partial L}{\partial o_t} \frac{\partial o_t}{\partial h_t}
+ \frac{\partial L}{\partial h_{t+1}} \frac{\partial h_{t+1}}{\partial h_t}
$$
其中：
- $\dfrac{\partial o_t}{\partial h_t} = W_{oh}^\top$
- $\dfrac{\partial h_{t+1}}{\partial h_t} = W_{hh}^\top$

记 $\delta_t = \dfrac{\partial L}{\partial h_t}$，则递推式：
$$
\delta_t = (o_t - y_t) W_{oh}^\top + \delta_{t+1} W_{hh}^\top
$$
边界条件：最后一步无后续状态，$\delta_T = (o_T - y_T) W_{oh}^\top$。

向前递推展开任意时刻 $t$ 的 $\delta_t$：
$$
\delta_t = (o_t-y_t)W_{oh}^\top
+ (o_{t+1}-y_{t+1})W_{oh}^\top W_{hh}^\top
+ (o_{t+2}-y_{t+2})W_{oh}^\top (W_{hh}^\top)^2
+ \dots
+ (o_T-y_T)W_{oh}^\top (W_{hh}^\top)^{T-t}
$$

**步骤3：求 $\dfrac{\partial h_t}{\partial W_{hh}}$**
$h_t = W_{hh}h_{t-1} + W_{hx}x_t$，对矩阵$W_{hh}$求导：
$$
\frac{\partial h_t}{\partial W_{hh}} = h_{t-1}^\top
$$

**步骤4：总梯度 $\dfrac{\partial L}{\partial W_{hh}}$**  
总损失为所有时间步累加，每个$h_t$都依赖$W_{hh}$：
$$
\frac{\partial L}{\partial W_{hh}}
= \sum_{t=1}^T \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}}
$$
代入$\delta_t = \dfrac{\partial L}{\partial h_t}$与$\dfrac{\partial h_t}{\partial W_{hh}}=h_{t-1}^\top$：
$$
\boldsymbol{\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \delta_t \, h_{t-1}^\top}
$$

把$\delta_t$完整展开，得到全时间步链式展开形式：
$$
\begin{aligned}
\frac{\partial L}{\partial W_{hh}}
=& \sum_{t=1}^T \Bigg[
\big(o_t-y_t\big)W_{oh}^\top
+ \sum_{k=t+1}^T \big(o_k-y_k\big)W_{oh}^\top \big(W_{hh}^\top\big)^{k-t}
\Bigg] h_{t-1}^\top
\end{aligned}
$$


**梯度消失/梯度爆炸产生条件**     
从$\delta_t$的展开式可见，梯度中存在**$W_{hh}$的幂次连乘项** $\displaystyle (W_{hh})^{k-t}$，由矩阵谱半径决定趋势：
设 $\rho(W_{hh})$ 为矩阵$W_{hh}$的**谱半径**（所有特征值模的最大值）。

1. **梯度爆炸**
若 $\boldsymbol{\rho(W_{hh}) > 1}$
序列越长，幂次$(W_{hh})^n$元素数值指数放大，梯度随序列长度急剧增大，出现梯度爆炸。

2. **梯度消失**
若 $\boldsymbol{\rho(W_{hh}) < 1}$
序列越长，幂次$(W_{hh})^n$元素指数趋近于0，远端时间步的梯度几乎无法传回前端，长距离依赖梯度消失。

3. 临界：$\rho(W_{hh})=1$，梯度不放大也不衰减。


### 2.2 编程题
实现一个简单的RNN单元的前向传播和单步反向传播（仅计算梯度，不更新）。给定输入 $x_t$ (形状(batch_size, input_size))、上一隐藏状态h_prev(形状(batch_size, hidden_size))，以及权重W_hx, W_hh, b_h，计算当前隐藏状态h_t。同时实现反向传播，已知上游梯度dh_next（即损失对h_t 的梯度），计算dx_t, dh_prev, dW_hx, dW_hh, db_h（使用 tanh 激活函数）。


In [17]:
import numpy as np

def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    x_t: (batch, input_size)
    h_prev: (batch, hidden_size)
    W_hx: (input_size, hidden_size)
    W_hh: (hidden_size, hidden_size)
    b_h: (hidden_size,)
    """
    a = x_t @ W_hx + h_prev @ W_hh + b_h   # (batch, hidden)
    h_t = np.tanh(a)
    cache = (x_t, h_prev, a, W_hx, W_hh)
    return h_t, cache

def rnn_step_backward(dh_next, cache):
    """ dh_next: (batch, hidden)，即 ∂L/∂h_t """
    x_t, h_prev, a, W_hx, W_hh = cache
    da = dh_next * (1 - np.tanh(a) ** 2)            # (batch, hidden)
    dx_t = da @ W_hx.T                               # (batch, input)
    dh_prev = da @ W_hh.T                            # (batch, hidden)
    dW_hx = x_t.T @ da                               # (input, hidden)
    dW_hh = h_prev.T @ da                            # (hidden, hidden)
    db_h = da.sum(axis=0)                            # (hidden,)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试
np.random.seed(42)
x = np.random.randn(3, 4)      # batch=3, input=4
h_prev = np.random.randn(3, 5) # hidden=5
W_hx = np.random.randn(4, 5)
W_hh = np.random.randn(5, 5)
b_h = np.random.randn(5)

h_t, cache = rnn_step_forward(x, h_prev, W_hx, W_hh, b_h)
dh_next = np.ones_like(h_t)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)
print("h_t shape:", h_t.shape)
print("dx_t shape:", dx_t.shape)
print("dW_hh shape:", dW_hh.shape)

h_t shape: (3, 5)
dx_t shape: (3, 4)
dW_hh shape: (5, 5)




## 三、 高级循环神经网络
### 3.1 理论计算题
假设一个深度双向RNN，有L层，每层隐藏单元数为H，输入维度为 D，输出维度为O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。


- **第一层（输入→隐藏）**：前向和反向各有一组参数，每组 $H \times D + H$，共 $2(H D + H)$。
- **第 $l$ 层（$l \ge 2$，隐藏→隐藏）**：每层前向和反向各有一组 $H \times H + H$，共 $(L-1)$ 层，参数数 $2(L-1)(H^2 + H)$。
- **输出层**：拼接前向和反向最后隐藏状态，输入维度 $2H$，输出 $O$，参数数 $O \times 2H + O$。

总参数数：
$$
\text{Total} = 2(H D + H) + 2(L-1)(H^2 + H) + (2H O + O)
$$
化简：
$$
= 2HD + 2H + 2(L-1)H^2 + 2(L-1)H + 2HO + O
$$
$$
= 2(L-1)H^2 + 2HD + 2HO + 2LH + O
$$




### 3.2 编程题
实现一个双向RNN 编码器，接收序列X(形状(seq_len, batch, input_dim))，使用torch.nn.RNN 或手动实现。要求返回每个时间步的拼接后的前向和后向隐藏状态(形状(seq_len, batch, 2*hidden_dim))，以及最终时间步的拼接隐藏状态（作为序列表示）。


In [5]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, rnn_type='rnn'):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        if rnn_type == 'rnn':
            self.rnn = nn.RNN(input_dim, hidden_dim, num_layers,
                              batch_first=False, bidirectional=True)
        elif rnn_type == 'lstm':
            self.rnn = nn.LSTM(input_dim, hidden_dim, num_layers,
                               batch_first=False, bidirectional=True)
        elif rnn_type == 'gru':
            self.rnn = nn.GRU(input_dim, hidden_dim, num_layers,
                              batch_first=False, bidirectional=True)
        else:
            raise ValueError("rnn_type must be 'rnn', 'lstm', or 'gru'")

    def forward(self, X):
        # X: (seq_len, batch, input_dim)
        outputs, hidden = self.rnn(X)  # outputs: (seq_len, batch, 2*hidden_dim)

        # 获取最后一层双向隐藏状态
        if isinstance(hidden, tuple):  # LSTM
            h, c = hidden
            last_layer_h = h[-2:, :, :]  # (2, batch, hidden_dim)
        else:
            last_layer_h = hidden[-2:, :, :]  # (2, batch, hidden_dim)

        final_state = torch.cat([last_layer_h[0], last_layer_h[1]], dim=-1)  # (batch, 2*hidden_dim)
        return outputs, final_state

In [6]:
##测试示例：
seq_len, batch, input_dim, hidden_dim = 10, 4, 16, 32
X = torch.randn(seq_len, batch, input_dim)
encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
outputs, final_state = encoder(X)
print(f"outputs shape: {outputs.shape}")      # (10, 4, 64)
print(f"final_state shape: {final_state.shape}")  # (4, 64)

outputs shape: torch.Size([10, 4, 64])
final_state shape: torch.Size([4, 64])



## 四、 嵌入向量
### 4.1 理论计算题
在Skip-gram 模型中，给定中心词 $w_{c}$ 和上下文词 $w_{o}$，使用负采样（采样 K 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $v_{c}$，$u_{o}$，负样本词向量为 $u_{n_{k}}$，写出完整的目标函数。


Skip-gram的原始目标（softmax）：
$$  
P(w_o \mid w_c) = \frac{\exp(\mathbf{u}_o^\top \mathbf{v}_c)}{\sum_{i=1}^V \exp(\mathbf{u}_i^\top \mathbf{v}_c)}
$$

负采样将问题转化为二分类，目标函数为：
$$
\mathcal{L} = -\log \sigma(\mathbf{u}_o^\top \mathbf{v}_c) - \sum_{k=1}^K \mathbb{E}_{n_k \sim P_n} \left[ \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_c) \right]
$$
其中 $\sigma(x) = 1/(1+e^{-x})$。

完整对数似然（最大化正样本概率，最小化负样本概率）：
$$
\mathcal{L} = -\log \sigma(\mathbf{u}_o^\top \mathbf{v}_c) - \sum_{k=1}^K \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_c)
$$

**噪声分布采样**：常用的是词频的 \(3/4\) 次方分布：
$$
P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{i=1}^V \text{count}(w_i)^{3/4}}
$$
该分布能提高低频词的采样概率。


### 4.2 编程题
实现CBOW 模型的前向传播和损失计算（不使用负采样，仅用完整softmax）。给定一批上下文词的索引列表（每个样本有context_size 个上下文词），词汇表大小 $V$，嵌入维度$d$。输入权重矩阵 W (形状(V, d))和输出权重矩阵 $W_{out}$ (形状(d, V))。计算平均上下文向量作为隐藏层，然后计算输出概率分布，并计算交叉熵损失（目标为中心词索引）。返回损失值。


In [7]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, W_in, W_out):
    """
    返回概率分布
    context_indices: (batch_size, context_size)
    W_in: (V, d)
    W_out: (d, V)
    """
    batch_size, context_size = context_indices.shape
    context_embeds = W_in[context_indices]          # (batch_size, context_size, d)
    h = torch.mean(context_embeds, dim=1)          # (batch_size, d)
    logits = torch.matmul(h, W_out)                # (batch_size, V)
    probs = F.softmax(logits, dim=-1)
    return probs

def cbow_loss(context_indices, target, W_in, W_out):
    """
    context_indices: (batch_size, context_size)
    target: (batch_size,)
    """
    probs = cbow_forward(context_indices, W_in, W_out)
    loss = F.cross_entropy(torch.log(probs + 1e-10), target)
    return loss

In [8]:
##测试示例：
V, d, batch_size, context_size = 1000, 64, 32, 4
W_in = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)
context_indices = torch.randint(0, V, (batch_size, context_size))
target = torch.randint(0, V, (batch_size,))

loss = cbow_loss(context_indices, target, W_in, W_out)
print(f"Loss: {loss.item():.4f}")
loss.backward()
print(f"W_in.grad shape: {W_in.grad.shape}, W_out.grad shape: {W_out.grad.shape}")


Loss: 13.8852
W_in.grad shape: torch.Size([1000, 64]), W_out.grad shape: torch.Size([64, 1000])



## 五、 注意力机制
### 5.1 理论计算题
给定查询矩阵 $Q \in \mathbb{R}^{2 ×4}$，键矩阵 $K \in \mathbb{R}^{3 ×4}$，值矩阵 $V \in \mathbb{R}^{3 ×5}$。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再softmax，再加权求和）。使用 $score =Q K^{T} / \sqrt{d_{k}}(d_{k}=4)$。可以只列出数值计算过程（用符号或具体数值）。


**步骤1：得分矩阵**
$$
S = \frac{Q K^T}{\sqrt{4}} = \frac{Q K^T}{2}
$$

**步骤2：softmax归一化（对每行）**
$$
A_{ij} = \text{softmax}(S_{i,:})_j = \frac{\exp(S_{ij})}{\sum_{k=1}^3 \exp(S_{ik})}
$$

**步骤3：加权求和**
$$
\text{Output} = A \cdot V \quad (\text{Output} \in \mathbb{R}^{2 \times 5})
$$

**数值示例（假设简化数据）**：

设
$$
Q = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix},\quad
K = \begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 0 & 1 & 1 \\ 1 & 0 & 1 & 0 \end{bmatrix},\quad
V = \begin{bmatrix} 1 & 2 & 3 & 4 & 5 \\ 5 & 4 & 3 & 2 & 1 \\ 2 & 3 & 4 & 5 & 6 \end{bmatrix}
$$


$$
Q K^T = \begin{bmatrix} 1 & 1 & 2 \\ 1 & 1 & 0 \end{bmatrix},\quad
S = \frac{1}{2} \begin{bmatrix} 1 & 1 & 2 \\ 1 & 1 & 0 \end{bmatrix} = \begin{bmatrix} 0.5 & 0.5 & 1 \\ 0.5 & 0.5 & 0 \end{bmatrix}
$$

softmax每行：
- 第1行：$[0.5, 0.5, 1] \to [0.212, 0.212, 0.576]$
- 第2行：$[0.5, 0.5, 0] \to [0.366, 0.366, 0.268]$

$$
A = \begin{bmatrix} 0.212 & 0.212 & 0.576 \\ 0.366 & 0.366 & 0.268 \end{bmatrix}
$$

输出：
$$
\text{Output} = A V = \begin{bmatrix}
0.212 \times V_1 + 0.212 \times V_2 + 0.576 \times V_3 \\
0.366 \times V_1 + 0.366 \times V_2 + 0.268 \times V_3
\end{bmatrix}
$$
计算得：
$$
= \begin{bmatrix}
2.424 & 3.424 & 4.424 & 5.424 & 6.424 \\
2.732 & 3.0 & 3.268 & 3.536 & 3.804
\end{bmatrix}
$$



### 5.2 编程题
实现多头注意力(Multi-Head Attention)的前向传播，假设 $num\_heads =2$，$d\_model =4$。给定输入 X (形状(seq_len, batch, d_model))，分别线性投影得到 Q , K ,V（每个头的维度 $d_k=d_v=d\_model / num\_heads$）。对每个头计算缩放点积注意力，然后将所有头的输出拼接并经过最终线性层。返回输出（形状与输入相同）。

In [18]:
import torch
import torch.nn.functional as F

def multi_head_attention(X, num_heads=2, d_model=4):
    """
    X: (seq_len, batch, d_model)
    """
    seq_len, batch, _ = X.shape
    d_k = d_model // num_heads   # 2

    # 线性投影权重（随机初始化，模拟学习参数）
    W_q = torch.randn(d_model, d_model) 
    W_k = torch.randn(d_model, d_model)
    W_v = torch.randn(d_model, d_model)
    W_o = torch.randn(d_model, d_model)
    
    Q = X @ W_q   # (seq, batch, d_model)
    K = X @ W_k
    V = X @ W_v

    def transpose_for_multihead(tensor):
        # (seq, batch, d_model) -> (batch*num_heads, seq, d_k)
        tensor = tensor.view(seq_len, batch, num_heads, d_k)
        tensor = tensor.permute(1, 2, 0, 3).contiguous()
        return tensor.view(batch * num_heads, seq_len, d_k)

    Q_mh = transpose_for_multihead(Q)
    K_mh = transpose_for_multihead(K)
    V_mh = transpose_for_multihead(V)

    # 缩放点积注意力
    scores = (Q_mh @ K_mh.transpose(-2, -1)) / (d_k ** 0.5)
    attn = F.softmax(scores, dim=-1)
    head_out = attn @ V_mh   # (batch*num_heads, seq, d_k)

    # 合并多头
    head_out = head_out.view(batch, num_heads, seq_len, d_k)
    head_out = head_out.permute(2, 0, 1, 3).contiguous()  # (seq, batch, num_heads, d_k)
    concat = head_out.view(seq_len, batch, d_model)       # 拼接

    output = concat @ W_o
    return output

# 测试
X = torch.randn(3, 2, 4)   # seq=3, batch=2
out = multi_head_attention(X)
print("多头注意力输出形状:", out.shape)  # (3, 2, 4)

多头注意力输出形状: torch.Size([3, 2, 4])
